# ESUS validation: 60 prompts

Runs `extract_arch.py` on `prompts/validation/prompt_01.txt` through `prompt_60.txt` and records timing, token use, and readiness classes.

Inputs expected under `/kaggle/input`:

- `esus-code/`: this repository snapshot (including `prompts/validation`);
- `qwen3-4b-instruct/`: `Qwen3-4B-Instruct-2507-Q4_K_M.gguf`.

The metadata file uses placeholders for the Kaggle username and dataset names.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, urllib.request, zipfile

INPUT = Path('/kaggle/input')
print('input tree:', [str(p) for p in sorted(INPUT.rglob('*'))[:30]])

def find_dir(name):
    return next((p for p in INPUT.rglob(name) if p.is_dir()), None)

def find_file(name):
    return next((p for p in INPUT.rglob(name) if p.is_file()), None)

CODE_ARCHIVE = find_file('esus-code.bundle') or find_file('esus-code.zip')
MODEL = find_file('Qwen3-4B-Instruct-2507-Q4_K_M.gguf')
WORK = Path('/kaggle/working/esus')
RUN_ID = time.strftime('kaggle-%Y%m%d-%H%M%S')

assert CODE_ARCHIVE is not None, 'missing esus-code.bundle in /kaggle/input'
assert MODEL is not None, 'missing Qwen GGUF in /kaggle/input'

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(CODE_ARCHIVE) as archive:
    archive.extractall(WORK)
os.chdir(WORK)
print('repo:', WORK)
print('model:', MODEL)
print('run id:', RUN_ID)


In [ ]:
# Prefer the official CUDA prebuilt; building llama.cpp from source is the
# fallback because Kaggle's CUDA 13 CMake target set is inconsistent.
server_bin = shutil.which('llama-server')
server_env = dict(os.environ)
if not server_bin:
    import tarfile, urllib.request
    release = 'b12110'
    base = 'https://github.com/ggml-org/llama.cpp/releases/download'
    root = Path('/kaggle/working/llama-prebuilt')
    root.mkdir(parents=True, exist_ok=True)
    assets = [
        f'llama-{release}-bin-ubuntu-cuda-12.8-x64.tar.gz',
        f'cudart-llama-{release}-bin-ubuntu-cuda-12.8-x64.tar.gz',
    ]
    for asset in assets:
        target = root / asset
        if not target.exists():
            print('downloading', asset, flush=True)
            urllib.request.urlretrieve(f'{base}/{release}/{asset}', target)
        with tarfile.open(target) as archive:
            archive.extractall(root)
    candidates = sorted(root.rglob('llama-server'))
    if not candidates:
        # Last-resort CPU build (slow, but keeps the validation run alive).
        build_dir = Path('/kaggle/working/llama.cpp')
        if not (build_dir / '.git').is_dir():
            subprocess.run(['git', 'clone', '--depth', '1',
                            'https://github.com/ggml-org/llama.cpp', str(build_dir)], check=True)
        subprocess.run(['cmake', '-S', str(build_dir), '-B', str(build_dir / 'build'),
                        '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF'], check=True)
        subprocess.run(['cmake', '--build', str(build_dir / 'build'),
                        '--config', 'Release', '-j2'], check=True)
        candidates = sorted((build_dir / 'build').rglob('llama-server'))
    assert candidates, 'llama-server binary not found'
    server_bin = str(candidates[0])
    lib_dirs = sorted({str(path.parent) for path in root.rglob('*.so*')})
    if lib_dirs:
        server_env['LD_LIBRARY_PATH'] = ':'.join(lib_dirs) + ':' + server_env.get('LD_LIBRARY_PATH', '')
assert Path(server_bin).is_file(), f'llama-server not found: {server_bin}'
print('llama-server:', server_bin)
print('LD_LIBRARY_PATH:', server_env.get('LD_LIBRARY_PATH', ''))


In [ ]:
# Start one llama-server per visible GPU, each pinned to its own device.
gpu_list = []
if shutil.which('nvidia-smi'):
    listing = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout
    gpu_list = [line for line in listing.splitlines() if line.startswith('GPU ')]
GPU_COUNT = max(1, min(2, len(gpu_list))) if gpu_list else 1
print('visible GPUs:', len(gpu_list), 'workers:', GPU_COUNT)

servers = []
for gpu in range(GPU_COUNT):
    port = 8080 + gpu
    log_path = Path(f'/kaggle/working/llama-server-{gpu}.log')
    log = open(log_path, 'w')
    env = dict(server_env)
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    args = [server_bin, '-m', str(MODEL)]
    if gpu_list:
        args += ['-ngl', '99']
    args += ['-c', '16384', '-fa', 'on', '--port', str(port), '-np', '1']
    print('starting GPU', gpu, 'port', port, flush=True)
    servers.append({
        'gpu': gpu, 'port': port, 'log': log_path,
        'proc': subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env),
    })

for item in servers:
    healthy = False
    for _ in range(240):
        if item['proc'].poll() is not None:
            break
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{item['port']}/health", timeout=2) as response:
                if response.status == 200:
                    healthy = True
                    break
        except Exception:
            time.sleep(2)
    if not healthy:
        tail = item['log'].read_text(errors='replace')[-4000:]
        for other in servers:
            if other['proc'].poll() is None:
                other['proc'].terminate()
        raise RuntimeError(f"server on GPU {item['gpu']} failed; log tail:\n{tail}")
    print('server ready on GPU', item['gpu'], 'port', item['port'], 'pid', item['proc'].pid)


In [ ]:
all_prompts = sorted(path.stem for path in (WORK / 'prompts' / 'validation').glob('prompt_*.txt'))
groups = [all_prompts[index::len(servers)] for index in range(len(servers))]
assert sorted(sum(groups, [])) == all_prompts, 'shards overlap or miss a prompt'
worker_runs = []
processes = []
for index, group in enumerate(groups):
    worker_id = f'{RUN_ID}-gpu{index}'
    worker_run = WORK / 'out' / 'validation' / 'runs' / worker_id
    worker_runs.append(worker_run)
    cmd = [sys.executable, 'scripts/run_validation.py',
           '--prompts', 'prompts/validation',
           '--out', str(worker_run.parent),
           '--run-id', worker_id,
           '--server-url', f"http://127.0.0.1:{servers[index]['port']}",
           '--only', ','.join(group),
           '--timeout', '1800']
    print(f"worker {index}: {len(group)} prompts -> {worker_run}", flush=True)
    log = open(f'/kaggle/working/worker-{index}.log', 'w')
    processes.append((index, log, subprocess.Popen(cmd, cwd=WORK, stdout=log, stderr=subprocess.STDOUT)))

failed = []
for index, log, process in processes:
    code = process.wait()
    log.flush()
    log.close()
    if code != 0:
        failed.append((index, code))
if failed:
    for index, code in failed:
        print(f'--- worker {index} log ---')
        print(Path(f'/kaggle/working/worker-{index}.log').read_text(errors='replace')[-5000:])
    raise RuntimeError(f'worker(s) failed: {failed}')

run_dir = WORK / 'out' / 'validation' / 'runs' / RUN_ID
merge = [sys.executable, 'scripts/merge_validation_runs.py',
         '--runs', *map(str, worker_runs),
         '--out', str(run_dir),
         '--require', str(len(all_prompts))]
print(' '.join(merge), flush=True)
subprocess.run(merge, cwd=WORK, check=True)
print('done:', run_dir)


In [ ]:
import csv
summary = run_dir / 'summary.csv'
with summary.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
print('prompts:', len(rows))
for row in rows:
    print(row['prompt'], row['classification'], row['seconds'],
          'out=' + row['tokens_out'], 'prompt=' + row['tokens_prompt'])

archive = shutil.make_archive(f'/kaggle/working/validation-{RUN_ID}', 'zip', run_dir)
print('archive:', archive)

## Notes

`scripts/run_validation.py` skips prompt directories with an existing `metrics.json` unless `--force` is used. To resume in a later Kaggle session, copy the previous `out/validation/runs/<run-id>` directory into the notebook or dataset and pass the same `--run-id`.

The classification and artifact definitions are documented in `doc/validation.md`.